# Delete `agent_chat_history.turns`

Companion to `app/gateway/chat_history/create_table.py`. Recreating this
table to add clustering (`user_id`, `conversation_id`) -- BigQuery doesn't
support adding clustering to an existing table via a plain `ALTER TABLE`;
doing it in place would need `bq update --clustering_fields=...` plus a
no-op `UPDATE ... SET col = col WHERE true` to actually recluster existing
rows, and that `UPDATE` would hit the same streaming-buffer restriction a
real `DELETE` would. Dropping and recreating is simpler here, and current
rows are disposable test data.

Prints the current row count first -- nothing here runs until you run it.

In [1]:
from google.cloud import bigquery

PROJECT_ID = "instacart-ml-model"
DATASET_ID = "agent_chat_history"
TABLE_ID = "turns"

client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

## Check what's there before deleting anything

In [4]:
try:
    table = client.get_table(table_ref)
    print(f"Table exists: {table_ref}")
    print(f"{len(table.schema)} top-level fields, {table.num_rows} rows (num_rows can lag for streamed data)")
except Exception as e:
    print(f"get_table failed -- table may not exist: {e}")

get_table failed -- table may not exist: 404 GET https://bigquery.googleapis.com/bigquery/v2/projects/instacart-ml-model/datasets/agent_chat_history/tables/turns?prettyPrint=false: Not found: Table instacart-ml-model:agent_chat_history.turns


## Delete

`not_found_ok=True` -- safe to re-run, matches `create_table.py`'s own
`exists_ok=True` convention.

In [3]:
client.delete_table(table_ref, not_found_ok=True)
print(f"Deleted (or already absent): {table_ref}")

Deleted (or already absent): instacart-ml-model.agent_chat_history.turns
